# Notebook 11 — Front Behavior Features

## Mục tiêu
Biến quỹ đạo sạch ở Notebook 10 thành đặc trưng hành vi theo **cửa sổ thời gian**.

Thiết kế mặc định:
- cửa sổ: **5 giây**
- bước trượt: **1 giây**
- dự đoán hành vi sau này có thể cập nhật mỗi 1 giây từ 5 giây dữ liệu gần nhất
- đồng thời tính `distance_last_1s_px` để hiển thị trực tiếp trên Raspberry Pi

Notebook tạo:
1. **individual features** theo từng Track ID / trajectory segment;
2. **group features** theo video và thời gian để phân tích nhiều cá.

Không gọi các đặc trưng này là “stress”.

In [1]:
# ============================================================
# 0. SETUP
# ============================================================
from pathlib import Path
import json, math, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root():
    candidates = [Path("/home/diy-hus/fish"), Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for p in candidates:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p.resolve()
    return Path("/home/diy-hus/fish").resolve()

PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "results/trajectory/front_cleaned_trajectories.csv"
RESULTS_DIR = PROJECT_ROOT / "results/behavior"
LOG_DIR = PROJECT_ROOT / "logs/behavior/FRONT_BEHAVIOR_FEATURES_001"
for p in [RESULTS_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
assert INPUT_PATH.exists(), f"Run Notebook 10 first: {INPUT_PATH}"

PROJECT_ROOT: /home/diy-hus/fish


In [2]:
# ============================================================
# 1. CONFIG
# ============================================================
EXPERIMENT_ID = "FRONT_BEHAVIOR_FEATURES_001"

WINDOW_SEC = 5.0
STEP_SEC = 1.0
TRAILING_DISTANCE_SEC = 1.0
MIN_WINDOW_COVERAGE = 0.60
IMMOBILE_SPEED_NORM_S = 0.01  # normalized image diagonal per second, initial threshold

INDIVIDUAL_FEATURES_PATH = RESULTS_DIR / "front_individual_behavior_features.csv"
GROUP_FEATURES_PATH = RESULTS_DIR / "front_group_behavior_features.csv"
FEATURE_SCHEMA_PATH = RESULTS_DIR / "front_behavior_feature_schema.json"

print("Window:", WINDOW_SEC, "s")
print("Step:", STEP_SEC, "s")

Window: 5.0 s
Step: 1.0 s


In [3]:
# ============================================================
# 2. LOAD CLEAN TRAJECTORIES
# ============================================================
traj = pd.read_csv(INPUT_PATH)

required = [
    "video_id","context","frame_index","time_sec","pred_track_id",
    "trajectory_uid","cx_clean","cy_clean","x_norm","y_norm",
    "observed","interpolated","fps_source","frame_width","frame_height",
]
missing = [c for c in required if c not in traj.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

traj = traj.sort_values(["video_id","trajectory_uid","frame_index"]).reset_index(drop=True)
traj["observed"] = traj["observed"].astype(bool)
traj["interpolated"] = traj["interpolated"].astype(bool)

print("Rows:", len(traj))
print("Trajectory UIDs:", traj["trajectory_uid"].nunique())
print("Videos:", traj["video_id"].value_counts().to_dict())

Rows: 40845
Trajectory UIDs: 286
Videos: {'V3_FEEDING': 20268, 'V5_NORMAL': 16507, 'V4_BREEDING': 2381, 'V8_PAIR_T1': 1689}


In [4]:
# ============================================================
# 3. FEATURE HELPERS
# ============================================================
def _step_kinematics(sub):
    sub = sub.sort_values("time_sec").copy()
    dt = sub["time_sec"].diff().to_numpy(float)
    dx = sub["cx_clean"].diff().to_numpy(float)
    dy = sub["cy_clean"].diff().to_numpy(float)

    step = np.hypot(dx, dy)
    speed = np.divide(step, dt, out=np.full_like(step, np.nan), where=dt > 0)

    diag = np.hypot(float(sub["frame_width"].iloc[0]), float(sub["frame_height"].iloc[0]))
    speed_norm = speed / diag if diag > 0 else np.full_like(speed, np.nan)

    angle = np.arctan2(dy, dx)
    dangle = np.diff(angle)
    dangle = (dangle + np.pi) % (2*np.pi) - np.pi

    return dt, dx, dy, step, speed, speed_norm, angle, dangle, diag

def _safe_mean(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return float(np.mean(x)) if len(x) else np.nan

def _safe_median(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return float(np.median(x)) if len(x) else np.nan

def _safe_std(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return float(np.std(x)) if len(x) else np.nan

def features_for_window(sub, start, end):
    w = sub[(sub["time_sec"] >= start) & (sub["time_sec"] < end)].sort_values("time_sec").copy()
    if len(w) < 2:
        return None

    fps = float(w["fps_source"].median())
    expected = max(1, int(round((end-start) * fps)))
    coverage = min(1.0, len(w) / expected)
    if coverage < MIN_WINDOW_COVERAGE:
        return None

    dt, dx, dy, step, speed, speed_norm, angle, dangle, diag = _step_kinematics(w)
    valid_step = step[np.isfinite(step)]
    distance = float(np.nansum(valid_step))

    net = float(np.hypot(
        w["cx_clean"].iloc[-1] - w["cx_clean"].iloc[0],
        w["cy_clean"].iloc[-1] - w["cy_clean"].iloc[0],
    ))
    efficiency = net / distance if distance > 1e-9 else 0.0

    accel = np.diff(speed)
    dt2 = dt[2:] if len(dt) >= 3 else np.array([])
    accel_rate = np.divide(
        accel[1:] if len(accel) >= 2 else np.array([]),
        dt2,
        out=np.full_like(dt2, np.nan, dtype=float),
        where=dt2 > 0,
    ) if len(dt2) else np.array([])

    # Distance in last 1 s of the behavior window.
    tail_start = max(start, end - TRAILING_DISTANCE_SEC)
    tail = w[w["time_sec"] >= tail_start]
    tail_step = np.hypot(tail["cx_clean"].diff(), tail["cy_clean"].diff())
    distance_1s = float(tail_step.fillna(0).sum())

    speed_norm_valid = speed_norm[np.isfinite(speed_norm)]
    immobile_ratio = float(np.mean(speed_norm_valid < IMMOBILE_SPEED_NORM_S)) if len(speed_norm_valid) else np.nan

    return {
        "window_start_sec": float(start),
        "window_end_sec": float(end),
        "window_mid_sec": float((start+end)/2),
        "n_points": int(len(w)),
        "coverage_ratio": float(coverage),
        "observed_ratio": float(w["observed"].mean()),
        "interpolated_ratio": float(w["interpolated"].mean()),
        "distance_5s_px": distance,
        "distance_last_1s_px": distance_1s,
        "net_displacement_px": net,
        "path_efficiency": float(efficiency),
        "mean_speed_px_s": _safe_mean(speed),
        "median_speed_px_s": _safe_median(speed),
        "max_speed_px_s": float(np.nanmax(speed)) if np.isfinite(speed).any() else np.nan,
        "speed_std_px_s": _safe_std(speed),
        "mean_speed_norm_s": _safe_mean(speed_norm),
        "max_speed_norm_s": float(np.nanmax(speed_norm)) if np.isfinite(speed_norm).any() else np.nan,
        "mean_abs_accel_px_s2": _safe_mean(np.abs(accel_rate)),
        "mean_abs_turn_rad": _safe_mean(np.abs(dangle)),
        "immobile_ratio": immobile_ratio,
        "x_mean_norm": float(w["x_norm"].mean()),
        "x_std_norm": float(w["x_norm"].std(ddof=0)),
        "x_range_norm": float(w["x_norm"].max() - w["x_norm"].min()),
        "y_mean_norm": float(w["y_norm"].mean()),
        "y_std_norm": float(w["y_norm"].std(ddof=0)),
        "y_range_norm": float(w["y_norm"].max() - w["y_norm"].min()),
        "bbox_area_mean_norm": float(
            (w["bbox_area"] / (w["frame_width"] * w["frame_height"])).mean()
        ) if "bbox_area" in w else np.nan,
    }

In [5]:
# ============================================================
# 4. INDIVIDUAL 5s WINDOW FEATURES, STEP 1s
# ============================================================
rows = []

for uid, sub in traj.groupby("trajectory_uid", sort=False):
    sub = sub.sort_values("time_sec")
    t0 = math.ceil(float(sub["time_sec"].min()) / STEP_SEC) * STEP_SEC
    t_last = float(sub["time_sec"].max())

    start = t0
    while start + WINDOW_SEC <= t_last + 1e-9:
        feat = features_for_window(sub, start, start + WINDOW_SEC)
        if feat is not None:
            first = sub.iloc[0]
            feat.update({
                "window_id": f"{uid}:W{int(round(start*1000)):09d}",
                "video_id": first["video_id"],
                "context": first["context"],
                "pred_track_id": int(first["pred_track_id"]),
                "trajectory_uid": uid,
            })
            rows.append(feat)
        start += STEP_SEC

IND = pd.DataFrame(rows)
if len(IND) == 0:
    raise RuntimeError("NO_INDIVIDUAL_FEATURE_WINDOWS")

IND = IND.sort_values(["video_id","trajectory_uid","window_start_sec"]).reset_index(drop=True)
IND.to_csv(INDIVIDUAL_FEATURES_PATH, index=False)

print("Individual windows:", len(IND))
print("By context:")
display(IND["context"].value_counts().rename_axis("context").reset_index(name="windows"))
print("Saved:", INDIVIDUAL_FEATURES_PATH)

Individual windows: 706
By context:


,context,windows
0,NORMAL,312
1,FEEDING,300
2,BREEDING_PARENTAL_CARE,69
3,PAIR_T1,25


Saved: /home/diy-hus/fish/results/behavior/front_individual_behavior_features.csv


In [6]:
# ============================================================
# 5. GROUP-LEVEL FEATURES, 5s WINDOW / 1s STEP
# ============================================================
group_rows = []

for video_id, sub_v in traj.groupby("video_id", sort=False):
    t0 = math.ceil(float(sub_v["time_sec"].min()) / STEP_SEC) * STEP_SEC
    t_last = float(sub_v["time_sec"].max())
    context = str(sub_v["context"].iloc[0])

    start = t0
    while start + WINDOW_SEC <= t_last + 1e-9:
        w = IND[
            IND["video_id"].eq(video_id)
            & IND["window_start_sec"].eq(float(start))
        ].copy()

        if len(w):
            group_rows.append({
                "group_window_id": f"{video_id}:GW{int(round(start*1000)):09d}",
                "video_id": video_id,
                "context": context,
                "window_start_sec": float(start),
                "window_end_sec": float(start + WINDOW_SEC),
                "active_tracks": int(w["trajectory_uid"].nunique()),
                "mean_distance_5s_px": float(w["distance_5s_px"].mean()),
                "median_distance_5s_px": float(w["distance_5s_px"].median()),
                "mean_distance_last_1s_px": float(w["distance_last_1s_px"].mean()),
                "mean_speed_px_s": float(w["mean_speed_px_s"].mean()),
                "mean_immobile_ratio": float(w["immobile_ratio"].mean()),
                "mean_y_norm": float(w["y_mean_norm"].mean()),
                "std_y_norm_between_tracks": float(w["y_mean_norm"].std(ddof=0)),
                "mean_x_norm": float(w["x_mean_norm"].mean()),
                "std_x_norm_between_tracks": float(w["x_mean_norm"].std(ddof=0)),
            })
        start += STEP_SEC

GROUP = pd.DataFrame(group_rows)
GROUP.to_csv(GROUP_FEATURES_PATH, index=False)

print("Group windows:", len(GROUP))
print("Saved:", GROUP_FEATURES_PATH)

Group windows: 312
Saved: /home/diy-hus/fish/results/behavior/front_group_behavior_features.csv


In [7]:
# ============================================================
# 6. FEATURE SCHEMA FOR NOTEBOOK 13 / RASPBERRY PI
# ============================================================
MODEL_FEATURE_COLUMNS = [
    "coverage_ratio","observed_ratio","interpolated_ratio",
    "distance_5s_px","distance_last_1s_px","net_displacement_px","path_efficiency",
    "mean_speed_px_s","median_speed_px_s","max_speed_px_s","speed_std_px_s",
    "mean_speed_norm_s","max_speed_norm_s","mean_abs_accel_px_s2","mean_abs_turn_rad",
    "immobile_ratio",
    "x_mean_norm","x_std_norm","x_range_norm",
    "y_mean_norm","y_std_norm","y_range_norm",
    "bbox_area_mean_norm",
]

schema = {
    "experiment_id": EXPERIMENT_ID,
    "window_sec": WINDOW_SEC,
    "step_sec": STEP_SEC,
    "trailing_distance_sec": TRAILING_DISTANCE_SEC,
    "feature_columns": MODEL_FEATURE_COLUMNS,
    "pi_realtime_fields": [
        "pred_track_id",
        "distance_last_1s_px",
        "mean_speed_px_s",
        "x_mean_norm",
        "y_mean_norm",
    ],
}
FEATURE_SCHEMA_PATH.write_text(json.dumps(schema, indent=2), encoding="utf-8")
print(json.dumps(schema, indent=2))

{
  "experiment_id": "FRONT_BEHAVIOR_FEATURES_001",
  "window_sec": 5.0,
  "step_sec": 1.0,
  "trailing_distance_sec": 1.0,
  "feature_columns": [
    "coverage_ratio",
    "observed_ratio",
    "interpolated_ratio",
    "distance_5s_px",
    "distance_last_1s_px",
    "net_displacement_px",
    "path_efficiency",
    "mean_speed_px_s",
    "median_speed_px_s",
    "max_speed_px_s",
    "speed_std_px_s",
    "mean_speed_norm_s",
    "max_speed_norm_s",
    "mean_abs_accel_px_s2",
    "mean_abs_turn_rad",
    "immobile_ratio",
    "x_mean_norm",
    "x_std_norm",
    "x_range_norm",
    "y_mean_norm",
    "y_std_norm",
    "y_range_norm",
    "bbox_area_mean_norm"
  ],
  "pi_realtime_fields": [
    "pred_track_id",
    "distance_last_1s_px",
    "mean_speed_px_s",
    "x_mean_norm",
    "y_mean_norm"
  ]
}


In [8]:
# ============================================================
# 7. QUICK QC
# ============================================================
display(IND[[
    "video_id","context","pred_track_id","window_start_sec",
    "distance_last_1s_px","distance_5s_px","mean_speed_px_s",
    "immobile_ratio","coverage_ratio"
]].head(20))

print("NaN rate in model features:")
display(IND[MODEL_FEATURE_COLUMNS].isna().mean().sort_values(ascending=False).rename("nan_rate").to_frame())

,video_id,context,pred_track_id,window_start_sec,distance_last_1s_px,distance_5s_px,mean_speed_px_s,immobile_ratio,coverage_ratio
0,V3_FEEDING,FEEDING,102,40.0,121.342336,415.356799,83.784261,0.134752,1.0
1,V3_FEEDING,FEEDING,102,41.0,47.619498,421.313499,84.985825,0.127660,1.0
2,V3_FEEDING,FEEDING,102,42.0,160.338775,489.086333,98.656715,0.106383,1.0
3,V3_FEEDING,FEEDING,102,43.0,155.084351,538.958070,108.716660,0.063830,1.0
4,V3_FEEDING,FEEDING,102,44.0,269.311341,772.811459,155.888714,0.035461,1.0
5,V3_FEEDING,FEEDING,102,45.0,129.889282,788.064862,157.846099,0.035211,1.0
6,V3_FEEDING,FEEDING,107,48.0,94.865541,737.430268,148.751749,0.035461,1.0
7,V3_FEEDING,FEEDING,107,49.0,63.857487,657.466812,132.621812,0.028369,1.0
8,V3_FEEDING,FEEDING,113,52.0,184.665217,698.427639,139.892138,0.070423,1.0
9,V3_FEEDING,FEEDING,113,53.0,249.732947,769.231679,155.166614,0.070922,1.0


NaN rate in model features:


,nan_rate
coverage_ratio,0.0
observed_ratio,0.0
interpolated_ratio,0.0
distance_5s_px,0.0
distance_last_1s_px,0.0
net_displacement_px,0.0
path_efficiency,0.0
mean_speed_px_s,0.0
median_speed_px_s,0.0
max_speed_px_s,0.0


In [9]:
# ============================================================
# 8. FINAL SUMMARY
# ============================================================
summary = {
    "experiment_id": EXPERIMENT_ID,
    "window_sec": WINDOW_SEC,
    "step_sec": STEP_SEC,
    "individual_windows": int(len(IND)),
    "group_windows": int(len(GROUP)),
    "feature_count": int(len(MODEL_FEATURE_COLUMNS)),
    "outputs": [
        str(INDIVIDUAL_FEATURES_PATH.relative_to(PROJECT_ROOT)),
        str(GROUP_FEATURES_PATH.relative_to(PROJECT_ROOT)),
        str(FEATURE_SCHEMA_PATH.relative_to(PROJECT_ROOT)),
    ],
}
(LOG_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("FINAL SUMMARY")
print(json.dumps(summary, indent=2))

FINAL SUMMARY
{
  "experiment_id": "FRONT_BEHAVIOR_FEATURES_001",
  "window_sec": 5.0,
  "step_sec": 1.0,
  "individual_windows": 706,
  "group_windows": 312,
  "feature_count": 23,
  "outputs": [
    "results/behavior/front_individual_behavior_features.csv",
    "results/behavior/front_group_behavior_features.csv",
    "results/behavior/front_behavior_feature_schema.json"
  ]
}
